In [6]:
import duckdb
from pathlib import Path

BASE = Path(r"C:\Users\blobf\OneDrive\MLB Clutch\mlb_clutch_data")

# Use an in-memory DuckDB connection.
# Your Parquet files remain the permanent data source.
con = duckdb.connect()

PA_GLOB = str(BASE / "pa" / "**" / "pa_*.parquet")

# See all PA files DuckDB finds
files = con.sql(
    f"SELECT file FROM glob('{PA_GLOB}') ORDER BY file"
).df()

print(f"Found {len(files)} PA files.")
display(files.head())

Found 1 PA files.


,file
0,C:\Users\blobf\OneDrive\MLB Clutch\mlb_clutch_...


In [22]:
# ============================================================
# CAREER NOTEBOOK — LOAD CHADWICK REGISTER
# ============================================================

from pathlib import Path
import pandas as pd
import requests

BASE = Path(
    r"C:\Users\blobf\OneDrive\MLB Clutch\mlb_clutch_data"
)

REGISTER_DIR = BASE / "chadwick_register"
REGISTER_DIR.mkdir(
    parents=True,
    exist_ok=True
)

REGISTER_URL = (
    "https://raw.githubusercontent.com/"
    "chadwickbureau/register/master/data/"
)

letters = list("0123456789abcdef")

register_parts = []

for letter in letters:

    filename = f"people-{letter}.csv"
    local_file = REGISTER_DIR / filename
    url = REGISTER_URL + filename

    if not local_file.exists():

        print(f"Downloading {filename}...")

        response = requests.get(
            url,
            timeout=60
        )

        response.raise_for_status()

        local_file.write_bytes(
            response.content
        )

    temp = pd.read_csv(
        local_file,
        low_memory=False
    )

    register_parts.append(temp)

    print(
        f"{filename}: {len(temp):,} rows"
    )

# ------------------------------------------------------------
# Combine Register
# ------------------------------------------------------------

chadwick = pd.concat(
    register_parts,
    ignore_index=True
)

# ------------------------------------------------------------
# Build player name
# ------------------------------------------------------------

chadwick["key_retro"] = (
    chadwick["key_retro"]
    .astype("string")
    .str.strip()
)

chadwick["key_mlbam"] = pd.to_numeric(
    chadwick["key_mlbam"],
    errors="coerce"
).astype("Int64")

chadwick["key_uuid"] = (
    chadwick["key_uuid"]
    .astype("string")
    .str.strip()
)

chadwick["player_name"] = (
    chadwick["name_first"]
    .fillna("")
    .astype(str)
    .str.strip()
    + " "
    + chadwick["name_last"]
    .fillna("")
    .astype(str)
    .str.strip()
).str.strip()

# Remove invalid names
chadwick.loc[
    chadwick["player_name"].isin(
        ["", "nan", "None"]
    ),
    "player_name"
] = pd.NA

# ------------------------------------------------------------
# Two separate lookup tables
# ------------------------------------------------------------

retro_lookup = (
    chadwick[
        [
            "key_retro",
            "key_uuid",
            "key_mlbam",
            "player_name"
        ]
    ]
    .dropna(subset=["key_retro"])
    .drop_duplicates(
        subset=["key_retro"],
        keep="first"
    )
)

mlbam_lookup = (
    chadwick[
        [
            "key_mlbam",
            "key_uuid",
            "player_name"
        ]
    ]
    .dropna(subset=["key_mlbam"])
    .drop_duplicates(
        subset=["key_mlbam"],
        keep="first"
    )
)

# ------------------------------------------------------------
# Register with DuckDB
# ------------------------------------------------------------

con.register(
    "retro_lookup_df",
    retro_lookup
)

con.register(
    "mlbam_lookup_df",
    mlbam_lookup
)

con.execute("""
CREATE OR REPLACE TEMP VIEW chadwick_retro AS
SELECT
    CAST(key_retro AS VARCHAR) AS key_retro,
    CAST(key_uuid AS VARCHAR) AS key_uuid,
    CAST(key_mlbam AS BIGINT) AS key_mlbam,
    CAST(player_name AS VARCHAR) AS player_name
FROM retro_lookup_df
""")

con.execute("""
CREATE OR REPLACE TEMP VIEW chadwick_mlbam AS
SELECT
    CAST(key_mlbam AS BIGINT) AS key_mlbam,
    CAST(key_uuid AS VARCHAR) AS key_uuid,
    CAST(player_name AS VARCHAR) AS player_name
FROM mlbam_lookup_df
""")

print()
print("Chadwick Register loaded.")
print(
    f"Retrosheet IDs: {len(retro_lookup):,}"
)
print(
    f"MLBAM IDs: {len(mlbam_lookup):,}"
)

people-0.csv: 32,500 rows
people-1.csv: 32,602 rows
people-2.csv: 32,321 rows
people-3.csv: 32,570 rows
people-4.csv: 32,482 rows
people-5.csv: 32,539 rows
people-6.csv: 32,362 rows
people-7.csv: 32,576 rows
people-8.csv: 32,278 rows
people-9.csv: 32,421 rows
people-a.csv: 32,937 rows
people-b.csv: 32,576 rows
people-c.csv: 32,556 rows
people-d.csv: 32,733 rows
people-e.csv: 32,459 rows
people-f.csv: 33,022 rows

Chadwick Register loaded.
Retrosheet IDs: 25,543
MLBAM IDs: 129,732


In [7]:
con.execute(f"""
CREATE OR REPLACE VIEW pa AS
SELECT *
FROM read_parquet(
    '{PA_GLOB}',
    union_by_name = true
)
""")

print("PA view created.")

PA view created.


In [23]:
# ============================================================
# CAREER PLAYER ROLE VIEW
# ============================================================

con.execute("""
CREATE OR REPLACE VIEW career_player_roles AS

SELECT
    season,
    game_date,
    game_id,

    inning,
    top_bot,
    outs_pre,

    run_diff,
    base_state,
    risp,

    is_pinch_hitter,

    pa_result,

    run_value,

    COALESCE(
        leverage_index,
        1.0
    ) AS leverage_index,

    situation_key,

    data_source,

    'hitter' AS role,

    batter_uuid AS person_uuid,
    batter_mlbam AS player_mlbam,
    batter_name AS player_name,

    run_value AS player_value

FROM pa

WHERE batter_name IS NOT NULL


UNION ALL


SELECT
    season,
    game_date,
    game_id,

    inning,
    top_bot,
    outs_pre,

    run_diff,
    base_state,
    risp,

    is_pinch_hitter,

    pa_result,

    run_value,

    COALESCE(
        leverage_index,
        1.0
    ) AS leverage_index,

    situation_key,

    data_source,

    'pitcher' AS role,

    pitcher_uuid AS person_uuid,
    pitcher_mlbam AS player_mlbam,
    pitcher_name AS player_name,

    -run_value AS player_value

FROM pa

WHERE pitcher_name IS NOT NULL
""")

print(
    "Career player-role view created."
)

NameError: name 'available_columns' is not defined

In [9]:
# ============================================================
# CAREER × SITUATION — ROBUST VERSION
# ============================================================

from pathlib import Path
import pandas as pd
import duckdb

BASE = Path(r"C:\Users\blobf\OneDrive\MLB Clutch\mlb_clutch_data")

# ------------------------------------------------------------
# 1. Read ALL PA parquet files directly
# ------------------------------------------------------------

PA_GLOB = str(BASE / "pa" / "**" / "pa_*.parquet")

pa_files = sorted(
    p for p in (BASE / "pa").rglob("pa_*.parquet")
    if p.is_file()
)

print(f"PA parquet files found: {len(pa_files)}")

if len(pa_files) == 0:
    raise FileNotFoundError(
        f"No PA parquet files found under:\n{BASE / 'pa'}"
    )

con.execute(f"""
CREATE OR REPLACE VIEW career_pa_raw AS
SELECT *
FROM read_parquet(
    '{PA_GLOB}',
    union_by_name = true
)
""")

pa_count = con.sql("""
    SELECT COUNT(*) AS n
    FROM career_pa_raw
""").fetchone()[0]

print(f"Total PA rows: {pa_count:,}")

# ------------------------------------------------------------
# 2. Inspect columns actually present
# ------------------------------------------------------------

cols = con.sql("""
    DESCRIBE career_pa_raw
""").df()["column_name"].tolist()

print("\nColumns available:")
print(cols)

# ------------------------------------------------------------
# 3. Find Chadwick Register files
# ------------------------------------------------------------

register_files = sorted(BASE.rglob("people-*.csv"))

print(f"\nChadwick Register files found: {len(register_files)}")

register = None

if register_files:
    pieces = []

    for f in register_files:
        try:
            x = pd.read_csv(f, low_memory=False)
            pieces.append(x)
        except Exception as e:
            print(f"Could not read {f}: {e}")

    if pieces:
        register = pd.concat(pieces, ignore_index=True)

        print(f"Register rows loaded: {len(register):,}")

        # Normalize important fields
        for c in [
            "key_retro",
            "key_mlbam",
            "key_uuid",
            "name_first",
            "name_last"
        ]:
            if c not in register.columns:
                register[c] = None

        register["key_retro"] = (
            register["key_retro"]
            .astype("string")
            .str.strip()
        )

        register["key_mlbam"] = (
            pd.to_numeric(
                register["key_mlbam"],
                errors="coerce"
            )
            .astype("Int64")
        )

        register["player_name"] = (
            register["name_first"]
            .fillna("")
            .astype(str)
            .str.strip()
            + " "
            + register["name_last"]
            .fillna("")
            .astype(str)
            .str.strip()
        ).str.strip()

        register.loc[
            register["player_name"].isin(["", "nan"]),
            "player_name"
        ] = None

        register = register.drop_duplicates(
            subset=["key_retro"],
            keep="first"
        )

        con.register("chadwick_df", register)

        con.execute("""
        CREATE OR REPLACE TEMP VIEW chadwick AS
        SELECT
            CAST(key_retro AS VARCHAR) AS key_retro,
            CAST(key_mlbam AS BIGINT) AS key_mlbam,
            CAST(key_uuid AS VARCHAR) AS key_uuid,
            player_name
        FROM chadwick_df
        """)

        has_chadwick = True

    else:
        has_chadwick = False

else:
    print("WARNING: Chadwick Register not found.")
    has_chadwick = False


# ------------------------------------------------------------
# 4. Determine the historical/current ID columns dynamically
# ------------------------------------------------------------

# Retrosheet batter ID
if "batter" in cols:
    batter_retro_expr = "CAST(batter AS VARCHAR)"
else:
    batter_retro_expr = "NULL"

# Retrosheet pitcher ID
if "pitcher" in cols:
    pitcher_retro_expr = "CAST(pitcher AS VARCHAR)"
else:
    pitcher_retro_expr = "NULL"

# MLBAM IDs from Statcast, if present
if "batter_mlbam" in cols:
    batter_mlbam_existing = "TRY_CAST(batter_mlbam AS BIGINT)"
else:
    batter_mlbam_existing = "NULL"

if "pitcher_mlbam" in cols:
    pitcher_mlbam_existing = "TRY_CAST(pitcher_mlbam AS BIGINT)"
else:
    pitcher_mlbam_existing = "NULL"

# Existing names, if present
if "batter_name" in cols:
    batter_name_existing = "CAST(batter_name AS VARCHAR)"
else:
    batter_name_existing = "NULL"

if "pitcher_name" in cols:
    pitcher_name_existing = "CAST(pitcher_name AS VARCHAR)"
else:
    pitcher_name_existing = "NULL"

# UUIDs, if already present
if "batter_uuid" in cols:
    batter_uuid_existing = "CAST(batter_uuid AS VARCHAR)"
else:
    batter_uuid_existing = "NULL"

if "pitcher_uuid" in cols:
    pitcher_uuid_existing = "CAST(pitcher_uuid AS VARCHAR)"
else:
    pitcher_uuid_existing = "NULL"

# ------------------------------------------------------------
# 5. Use available game-state columns
# ------------------------------------------------------------

def sql_col(name, fallback="NULL"):
    return f"CAST({name} AS VARCHAR)" if name in cols else fallback

inning_expr = sql_col("inning")
top_bot_expr = sql_col("top_bot")
outs_expr = sql_col("outs_pre")
run_diff_expr = sql_col("run_diff")

# run_diff may not exist in older files, so derive it if scores exist
if "run_diff" in cols:
    run_diff_expr = "TRY_CAST(run_diff AS DOUBLE)"
elif "score_v" in cols and "score_h" in cols:
    if "top_bot" in cols:
        run_diff_expr = """
        CASE
            WHEN CAST(top_bot AS VARCHAR) IN ('0', 'T', 'top')
                THEN TRY_CAST(score_v AS DOUBLE) - TRY_CAST(score_h AS DOUBLE)
            ELSE TRY_CAST(score_h AS DOUBLE) - TRY_CAST(score_v AS DOUBLE)
        END
        """
    else:
        run_diff_expr = "0"
else:
    run_diff_expr = "0"

base_state_expr = (
    "CAST(base_state AS VARCHAR)"
    if "base_state" in cols
    else "'000'"
)

risp_expr = (
    "CAST(risp AS BOOLEAN)"
    if "risp" in cols
    else f"""
    CASE
        WHEN {base_state_expr} IN ('010','001','110','101','011','111')
        THEN TRUE
        ELSE FALSE
    END
    """
)

run_value_expr = (
    "TRY_CAST(run_value AS DOUBLE)"
    if "run_value" in cols
    else None
)

if run_value_expr is None:
    raise ValueError(
        "The PA files do not contain 'run_value'. "
        "We need the run-value field before calculating clutch."
    )

leverage_expr = (
    "COALESCE(TRY_CAST(leverage_index AS DOUBLE), 1.0)"
    if "leverage_index" in cols
    else "1.0"
)

pinch_expr = (
    "CAST(is_pinch_hitter AS BOOLEAN)"
    if "is_pinch_hitter" in cols
    else "FALSE"
)

pa_result_expr = (
    "CAST(pa_result AS VARCHAR)"
    if "pa_result" in cols
    else "NULL"
)

season_expr = (
    "CAST(season AS INTEGER)"
    if "season" in cols
    else "NULL"
)

game_date_expr = (
    "CAST(game_date AS DATE)"
    if "game_date" in cols
    else "NULL"
)

game_id_expr = (
    "CAST(game_id AS VARCHAR)"
    if "game_id" in cols
    else "NULL"
)

# ------------------------------------------------------------
# 6. Build player roles
# ------------------------------------------------------------

if has_chadwick:

    hitter_name_expr = f"""
        COALESCE(
            NULLIF({batter_name_existing}, ''),
            h.player_name,
            {batter_retro_expr}
        )
    """

    pitcher_name_expr = f"""
        COALESCE(
            NULLIF({pitcher_name_existing}, ''),
            p.player_name,
            {pitcher_retro_expr}
        )
    """

    hitter_mlbam_expr = f"""
        COALESCE(
            {batter_mlbam_existing},
            h.key_mlbam
        )
    """

    pitcher_mlbam_expr = f"""
        COALESCE(
            {pitcher_mlbam_existing},
            p.key_mlbam
        )
    """

    hitter_uuid_expr = f"""
        COALESCE(
            {batter_uuid_existing},
            h.key_uuid
        )
    """

    pitcher_uuid_expr = f"""
        COALESCE(
            {pitcher_uuid_existing},
            p.key_uuid
        )
    """

    hitter_join = f"""
        LEFT JOIN chadwick h
            ON h.key_retro = {batter_retro_expr}
    """

    pitcher_join = f"""
        LEFT JOIN chadwick p
            ON p.key_retro = {pitcher_retro_expr}
    """

else:

    hitter_name_expr = (
        f"COALESCE(NULLIF({batter_name_existing}, ''), "
        f"{batter_retro_expr})"
    )

    pitcher_name_expr = (
        f"COALESCE(NULLIF({pitcher_name_existing}, ''), "
        f"{pitcher_retro_expr})"
    )

    hitter_mlbam_expr = batter_mlbam_existing
    pitcher_mlbam_expr = pitcher_mlbam_existing

    hitter_uuid_expr = batter_uuid_existing
    pitcher_uuid_expr = pitcher_uuid_existing

    hitter_join = ""
    pitcher_join = ""


# ------------------------------------------------------------
# 7. Create career player-role observations
# ------------------------------------------------------------

con.execute(f"""
CREATE OR REPLACE TEMP VIEW career_player_roles AS

SELECT
    {season_expr} AS season,
    {game_date_expr} AS game_date,
    {game_id_expr} AS game_id,

    TRY_CAST({inning_expr} AS INTEGER) AS inning,
    {top_bot_expr} AS top_bot,
    TRY_CAST({outs_expr} AS INTEGER) AS outs_pre,
    {run_diff_expr} AS run_diff,

    {base_state_expr} AS base_state,
    {risp_expr} AS risp,

    {pinch_expr} AS is_pinch_hitter,
    {pa_result_expr} AS pa_result,

    {run_value_expr} AS run_value,
    {leverage_expr} AS leverage_index,

    {hitter_uuid_expr} AS person_uuid,
    {hitter_mlbam_expr} AS player_mlbam,
    {hitter_name_expr} AS player_name,

    'hitter' AS role,

    {run_value_expr} AS player_value

FROM career_pa_raw
{hitter_join}

UNION ALL

SELECT
    {season_expr} AS season,
    {game_date_expr} AS game_date,
    {game_id_expr} AS game_id,

    TRY_CAST({inning_expr} AS INTEGER) AS inning,
    {top_bot_expr} AS top_bot,
    TRY_CAST({outs_expr} AS INTEGER) AS outs_pre,
    {run_diff_expr} AS run_diff,

    {base_state_expr} AS base_state,
    {risp_expr} AS risp,

    {pinch_expr} AS is_pinch_hitter,
    {pa_result_expr} AS pa_result,

    {run_value_expr} AS run_value,
    {leverage_expr} AS leverage_index,

    {pitcher_uuid_expr} AS person_uuid,
    {pitcher_mlbam_expr} AS player_mlbam,
    {pitcher_name_expr} AS player_name,

    'pitcher' AS role,

    -{run_value_expr} AS player_value

FROM career_pa_raw
{pitcher_join}
""")


# ------------------------------------------------------------
# 8. Verify player-role rows BEFORE grouping
# ------------------------------------------------------------

role_check = con.sql("""
SELECT
    role,
    COUNT(*) AS observations,
    COUNT(player_name) AS named_observations,
    COUNT(DISTINCT player_name) AS unique_players
FROM career_player_roles
GROUP BY role
ORDER BY role
""").df()

print("\nCAREER PLAYER ROLE CHECK")
display(role_check)


# ------------------------------------------------------------
# 9. Build the Career × Situation table
# ------------------------------------------------------------

career_situation = con.sql("""
WITH x AS (
    SELECT
        *,
        
        CASE
            WHEN inning <= 3 THEN 'Early'
            WHEN inning <= 6 THEN 'Middle'
            WHEN inning >= 7 THEN 'Late'
            ELSE 'Unknown'
        END AS inning_group,

        CASE
            WHEN run_diff = 0 THEN 'Tied'
            WHEN ABS(run_diff) = 1 THEN 'One-run game'
            WHEN ABS(run_diff) = 2 THEN 'Two-run game'
            WHEN ABS(run_diff) >= 3 THEN '3+ run game'
            ELSE 'Unknown'
        END AS score_situation,

        CASE
            WHEN outs_pre = 0 THEN '0 outs'
            WHEN outs_pre = 1 THEN '1 out'
            WHEN outs_pre = 2 THEN '2 outs'
            ELSE 'Unknown'
        END AS outs_situation,

        CASE
            WHEN base_state = '000' THEN 'Bases empty'
            WHEN base_state = '100' THEN 'Runner on 1st'
            WHEN base_state = '010' THEN 'Runner on 2nd'
            WHEN base_state = '001' THEN 'Runner on 3rd'
            WHEN base_state = '110' THEN 'Runners on 1st & 2nd'
            WHEN base_state = '101' THEN 'Runners on 1st & 3rd'
            WHEN base_state = '011' THEN 'Runners on 2nd & 3rd'
            WHEN base_state = '111' THEN 'Bases loaded'
            ELSE 'Unknown'
        END AS runner_situation,

        CASE
            WHEN ABS(run_diff) <= 1 THEN TRUE
            ELSE FALSE
        END AS close_game,

        CASE
            WHEN inning >= 7 THEN TRUE
            ELSE FALSE
        END AS late_game,

        CASE
            WHEN inning >= 7
                 AND ABS(run_diff) <= 1
            THEN TRUE
            ELSE FALSE
        END AS late_close_game

    FROM career_player_roles
)

SELECT
    role,
    player_name,
    player_mlbam,

    inning_group,
    score_situation,
    runner_situation,
    outs_situation,
    risp,
    close_game,
    late_game,
    late_close_game,

    COUNT(*) AS pa,

    AVG(player_value) AS average_value,

    SUM(player_value) AS total_value,

    AVG(leverage_index) AS average_leverage,

    SUM(player_value * leverage_index)
        / NULLIF(SUM(leverage_index), 0)
        AS weighted_situation_value

FROM x

GROUP BY
    role,
    player_name,
    player_mlbam,
    inning_group,
    score_situation,
    runner_situation,
    outs_situation,
    risp,
    close_game,
    late_game,
    late_close_game

ORDER BY
    role,
    player_name,
    pa DESC
""").df()

print(f"\nCAREER × SITUATION rows: {len(career_situation):,}")

display(career_situation.head(50))

PA parquet files found: 1
Total PA rows: 187,235

Columns available:
['game_pk', 'at_bat_number', 'game_date', 'pitch_number', 'batter', 'pitcher', 'inning', 'inning_topbot', 'outs_when_up', 'on_1b', 'on_2b', 'on_3b', 'home_score', 'away_score', 'bat_score', 'fld_score', 'pa_result', 'pa_description', 'delta_run_exp', 'delta_home_win_exp', 'season', 'batter_mlbam', 'pitcher_mlbam', 'top_bot', 'outs_pre', 'score_h', 'score_v', 'bat_score_before', 'fld_score_before', 'run_diff', 'br1_pre', 'br2_pre', 'br3_pre', 'base_state', 'risp', 'inning_bucket', 'score_diff_clamped', 'situation_key', 'run_value', 'pa_wpa', 'game_id', 'data_source', 'pbp_quality', 'batter_uuid', 'batter_name', 'pitcher_uuid', 'pitcher_name', 'leverage_index', 'is_pinch_hitter', 'pinch_hitter_source']

Chadwick Register files found: 0

CAREER PLAYER ROLE CHECK


,role,observations,named_observations,unique_players
0,hitter,187235,187235,1406
1,pitcher,187235,187235,1188



CAREER × SITUATION rows: 127,141


,role,player_name,player_mlbam,inning_group,score_situation,runner_situation,outs_situation,risp,close_game,late_game,late_close_game,pa,average_value,total_value,average_leverage,weighted_situation_value
0,hitter,457705,457705,Early,Tied,Bases empty,1 out,False,True,False,False,6,-0.226833,-1.361,1.0,-0.226833
1,hitter,457705,457705,Late,3+ run game,Bases empty,1 out,False,False,True,False,5,0.079000,0.395,1.0,0.079000
2,hitter,457705,457705,Early,Tied,Bases empty,0 outs,False,True,False,False,4,-0.148500,-0.594,1.0,-0.148500
3,hitter,457705,457705,Late,3+ run game,Bases empty,0 outs,False,False,True,False,4,-0.226500,-0.906,1.0,-0.226500
4,hitter,457705,457705,Early,Tied,Bases empty,2 outs,False,True,False,False,3,-0.079333,-0.238,1.0,-0.079333
5,hitter,457705,457705,Middle,3+ run game,Runner on 1st,0 outs,False,False,False,False,3,0.154667,0.464,1.0,0.154667
6,hitter,457705,457705,Middle,One-run game,Bases empty,1 out,False,True,False,False,3,-0.015000,-0.045,1.0,-0.015000
7,hitter,457705,457705,Middle,3+ run game,Bases empty,0 outs,False,False,False,False,3,-0.108667,-0.326,1.0,-0.108667
8,hitter,457705,457705,Middle,3+ run game,Bases empty,1 out,False,False,False,False,3,-0.078333,-0.235,1.0,-0.078333
9,hitter,457705,457705,Early,One-run game,Bases empty,1 out,False,True,False,False,3,-0.060000,-0.180,1.0,-0.060000


In [12]:
career_situation.to_csv(
    BASE
    / "summaries"
    / "career_clutch_by_situation.csv",
    index=False
)

In [13]:
# ============================================================
# LEVERAGE BANDS
# ============================================================

con.execute("""
CREATE OR REPLACE VIEW career_with_leverage_band AS

SELECT

    *,

    CASE

        WHEN leverage_index < 0.75
            THEN 'Low'

        WHEN leverage_index < 1.50
            THEN 'Normal'

        WHEN leverage_index < 2.50
            THEN 'High'

        ELSE 'Very High'

    END AS leverage_band

FROM career_player_roles
""")

In [15]:
con.sql("""
SELECT

    player_name,
    role,

    leverage_band,

    COUNT(*) AS pa,

    AVG(player_value)
        AS avg_value,

    AVG(leverage_index)
        AS avg_leverage

FROM career_with_leverage_band

WHERE player_name = 'Aaron Judge'

GROUP BY

    player_name,
    role,
    leverage_band

ORDER BY
    avg_leverage
""").df()

,player_name,role,leverage_band,pa,avg_value,avg_leverage


In [16]:
# ============================================================
# DESCRIPTIVE SITUATION FLAGS
# ============================================================

con.execute("""
CREATE OR REPLACE VIEW career_situation_flags AS

SELECT

    *,

    CASE
        WHEN ABS(run_diff) <= 1
        THEN TRUE
        ELSE FALSE
    END AS close_game,

    CASE
        WHEN run_diff = 0
        THEN TRUE
        ELSE FALSE
    END AS tied_game,

    CASE
        WHEN inning >= 7
        THEN TRUE
        ELSE FALSE
    END AS late_game,

    CASE
        WHEN inning >= 7
             AND ABS(run_diff) <= 1
        THEN TRUE
        ELSE FALSE
    END AS late_close_game

FROM career_player_roles
""")

In [17]:
con.sql("""
SELECT

    player_name,
    role,

    COUNT(*) AS pa,

    AVG(player_value)
        AS avg_value,

    AVG(leverage_index)
        AS avg_leverage

FROM career_situation_flags

WHERE
    role = 'hitter'
    AND risp = TRUE

GROUP BY

    player_name,
    role

HAVING COUNT(*) >= 100
""").df()

,player_name,role,pa,avg_value,avg_leverage
0,702222,hitter,118,0.079381,1.0
1,687401,hitter,125,0.007096,1.0
2,677594,hitter,142,-0.031134,1.0
3,669003,hitter,149,0.040456,1.0
4,571448,hitter,168,0.013887,1.0
...,...,...,...,...,...
205,691718,hitter,175,0.098000,1.0
206,671976,hitter,123,0.015220,1.0
207,677951,hitter,138,0.036428,1.0
208,665750,hitter,121,0.050479,1.0


In [18]:
con.sql("""
SELECT

    player_name,
    role,

    COUNT(*) AS pa,

    AVG(player_value)
        AS avg_value

FROM career_situation_flags

WHERE
    role = 'hitter'
    AND late_close_game = TRUE

GROUP BY

    player_name,
    role

HAVING COUNT(*) >= 50
""").df()

,player_name,role,pa,avg_value
0,695657,hitter,67,0.007642
1,665862,hitter,65,0.056123
2,683002,hitter,67,0.032433
3,656941,hitter,66,0.173591
4,686797,hitter,67,0.022925
...,...,...,...,...
153,672640,hitter,64,0.101016
154,802139,hitter,70,0.025800
155,676475,hitter,77,-0.095338
156,701538,hitter,68,0.082265


In [19]:
# ============================================================
# CAREER PINCH-HITTING VALUE
# ============================================================

pinch_hitting = con.sql("""
SELECT

    player_name,

    player_mlbam,

    COUNT(*) AS pinch_hit_pa,

    AVG(player_value)
        AS pinch_hit_value_per_pa,

    AVG(leverage_index)
        AS pinch_hit_mean_leverage,

    SUM(
        player_value
        *
        leverage_index
    )
    /
    NULLIF(
        SUM(leverage_index),
        0
    )
        AS pinch_hit_weighted_value

FROM career_player_roles

WHERE

    role = 'hitter'

    AND is_pinch_hitter = TRUE

GROUP BY

    player_name,

    player_mlbam

HAVING COUNT(*) >= 10

ORDER BY
    pinch_hit_weighted_value DESC
""").df()

display(
    pinch_hitting.head(25)
)

,player_name,player_mlbam,pinch_hit_pa,pinch_hit_value_per_pa,pinch_hit_mean_leverage,pinch_hit_weighted_value
0,605353,605353,10,0.177300,1.0,0.177300
1,656976,656976,12,0.161583,1.0,0.161583
2,703197,703197,25,0.132040,1.0,0.132040
3,806956,806956,14,0.125929,1.0,0.125929
4,680664,680664,119,0.094294,1.0,0.094294
5,694224,694224,11,0.090091,1.0,0.090091
6,686894,686894,102,0.074853,1.0,0.074853
7,691002,691002,27,0.073778,1.0,0.073778
8,687952,687952,198,0.069394,1.0,0.069394
9,664034,664034,468,0.067976,1.0,0.067976


In [21]:
# ============================================================
# PLAYER CAREER BY SEASON
# ============================================================

career_by_season = con.sql("""
WITH x AS (

    SELECT
        *,

        -- Create a stable player key for the calculation.
        -- Prefer UUID, then MLBAM ID, then player name.
        COALESCE(
            NULLIF(CAST(person_uuid AS VARCHAR), ''),
            CASE
                WHEN player_mlbam IS NOT NULL
                THEN CAST(player_mlbam AS VARCHAR)
            END,
            NULLIF(player_name, '')
        ) AS career_player_key

    FROM career_player_roles
),

with_career_baseline AS (

    SELECT

        x.*,

        -- Career baseline for this specific player and role
        AVG(player_value) OVER (
            PARTITION BY
                career_player_key,
                role
        ) AS career_baseline

    FROM x
)

SELECT

    player_name,
    player_mlbam,
    person_uuid,
    role,

    season,

    COUNT(*) AS pa,

    -- Ordinary season performance
    AVG(player_value)
        AS season_value_per_pa,

    SUM(player_value)
        AS season_total_value,

    -- Average importance of the situations faced that season
    AVG(leverage_index)
        AS season_average_leverage,

    -- Importance-weighted performance
    SUM(
        player_value * leverage_index
    )
    /
    NULLIF(
        SUM(leverage_index),
        0
    )
        AS season_weighted_value,

    -- Career-baseline-adjusted clutch contribution
    SUM(
        (
            player_value
            - career_baseline
        )
        * leverage_index
    )
        AS season_clutch_contribution,

    -- Average PA-level clutch differential in that season
    SUM(
        (
            player_value
            - career_baseline
        )
        * leverage_index
    )
    /
    NULLIF(
        SUM(leverage_index),
        0
    )
        AS season_clutch_value

FROM with_career_baseline

GROUP BY

    player_name,
    player_mlbam,
    person_uuid,
    role,
    season

ORDER BY

    player_name,
    season,
    role
""").df()

print(
    f"Career player-season rows: "
    f"{len(career_by_season):,}"
)

display(
    career_by_season.head(100)
)
career_by_season.to_csv(
    OUTPUT_DIR / "career_by_season.csv",
    index=False
)

print(
    "Saved:",
    OUTPUT_DIR / "career_by_season.csv"
)

Career player-season rows: 2,594


,player_name,player_mlbam,person_uuid,role,season,pa,season_value_per_pa,season_total_value,season_average_leverage,season_weighted_value,season_clutch_contribution,season_clutch_value
0,434378,434378,None,pitcher,2026,60,-0.087383,-5.243,1.0,-0.087383,1.110223e-15,1.850372e-17
1,445276,445276,None,pitcher,2026,182,0.033462,6.090,1.0,0.033462,-1.471046e-15,-8.082668e-18
2,453286,453286,None,pitcher,2026,334,-0.015757,-5.263,1.0,-0.015757,-3.774758e-15,-1.130167e-17
3,455119,455119,None,pitcher,2026,78,-0.108513,-8.464,1.0,-0.108513,2.442491e-15,3.131398e-17
4,457705,457705,None,hitter,2026,97,-0.050969,-4.944,1.0,-0.050969,2.220446e-16,2.289120e-18
...,...,...,...,...,...,...,...,...,...,...,...,...
95,592450,592450,None,hitter,2026,294,0.058772,17.279,1.0,0.058772,1.210143e-14,4.116133e-17
96,592454,592454,None,pitcher,2026,51,-0.094804,-4.835,1.0,-0.094804,2.331468e-15,4.571507e-17
97,592518,592518,None,hitter,2026,664,0.001161,0.771,1.0,0.001161,4.329870e-15,6.520888e-18
98,592626,592626,None,hitter,2026,449,0.032984,14.810,1.0,0.032984,-2.309264e-14,-5.143127e-17
